# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [ ]:
import json
import sqlite3
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# Robust pack discovery for local workspace and Colab runs
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif (Path.cwd() / "HW_2" / "flasheats-classroom-pack" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "HW_2" / "flasheats-classroom-pack"
elif (Path.cwd().parent / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent
elif Path("/content/flasheats_class6/database/flasheats.db").exists():
    BASE = Path("/content/flasheats_class6")
else:
    for candidate in [Path.cwd(), Path.cwd().parent, Path("/content")]:
        for match in candidate.rglob("flasheats.db"):
            BASE = match.parent.parent
            break
        if BASE is not None:
            break

if BASE is None or not (BASE / "database" / "flasheats.db").exists():
    raise FileNotFoundError("flasheats.db not found. Ensure the classroom pack is extracted.")

DB_PATH = BASE / "database" / "flasheats.db"
print("Detected BASE:", BASE)
print("Database exists:", DB_PATH.exists())


## Load the updated FlashEats data

In [ ]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

# Challenge 1 — Can we defend the “56% late” claim?

Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` is unique across all rows | Count unique `order_id` vs `len(orders)` | **WARN**: Overcounts duplicate order outcomes. |
| Delivered orders have completion time | If `final_status == 'delivered'`, `actual_delivery_at` is non-null | Check nulls on delivered rows | **FAIL**: Distorts late rate calculation and hides drops. |
| Promised ETA is valid | `promised_eta >= created_at` | Compare datetime values | **FAIL**: Negative target durations corrupt delivery SLAs. |
| Event chronology is valid | `pickup_at <= actual_delivery_at` | Compare datetime values | **FAIL**: Negative transit durations invalidate delivery timelines. |
| “Late” has an agreed definition | Single canonical rule across stakeholders | Audit `client_metric_definitions.json` | **FAIL**: Disparate definitions (56% vs 23%) cause organizational misalignment. |

**FDE Principle:** Never clean by instinct. First determine whether the data reliably supports the decision.


In [ ]:
# Challenge 1: Data validation checks

print("Total order rows:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())

# 1. Establish business grain & check duplicate rows
duplicate_mask = orders["order_id"].duplicated(keep=False)
duplicate_orders = orders[duplicate_mask]
print(f"Duplicate order rows: {len(duplicate_orders)} rows across {duplicate_orders['order_id'].nunique()} unique IDs")
display(duplicate_orders[["order_id", "customer_id", "created_at", "final_status", "traffic_bucket"]])

# Deduplicate on order_id keeping first
orders_dedup = orders.drop_duplicates(subset=["order_id"], keep="first").copy()

# Parse timestamps with mixed formatting
for col in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    orders_dedup[col + "_dt"] = pd.to_datetime(orders_dedup[col], format="mixed", errors="coerce")

# 2. Check delivered orders with missing completion time
orders_dedup["final_status_norm"] = orders_dedup["final_status"].astype(str).str.strip().str.lower()
delivered_orders = orders_dedup[orders_dedup["final_status_norm"] == "delivered"]
missing_completion = delivered_orders[delivered_orders["actual_delivery_at_dt"].isna()]
print(f"\nDelivered orders missing actual_delivery_at: {len(missing_completion)} / {len(delivered_orders)}")

# 3. Test promised_eta >= created_at
invalid_eta = orders_dedup[orders_dedup["promised_eta_dt"] < orders_dedup["created_at_dt"]]
print(f"Orders where promised_eta < created_at (anomaly): {len(invalid_eta)} / {len(orders_dedup)}")
if len(invalid_eta) > 0:
    display(invalid_eta[["order_id", "created_at", "promised_eta", "actual_delivery_at"]])

# 4. Test pickup_at <= actual_delivery_at
invalid_transit = orders_dedup[orders_dedup["actual_delivery_at_dt"] < orders_dedup["pickup_at_dt"]]
print(f"Orders where actual_delivery_at < pickup_at (anomaly): {len(invalid_transit)} / {len(orders_dedup)}")
if len(invalid_transit) > 0:
    display(invalid_transit[["order_id", "pickup_at", "actual_delivery_at", "final_status"]])


# Challenge 2 — Stakeholders disagree on “late”

Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|
| **VP Operations (>0m, Observed)** | **56.39%** | Strict contractual SLA ($843 / 1,495$); counts 30-second delays as operational failures. |
| **Support Lead (>10m, Observed)** | **23.34%** | Meaningful customer dissatisfaction threshold ($349 / 1,495$); aligns with ticket escalations. |
| **VP Operations (All Orders)** | **52.69%** | Measures lateness over entire order intake ($843 / 1,600$), diluting with cancellations. |
| **Finance (Delivered Only)** | **56.39%** | Evaluates settled revenue fulfillment; excludes 68 cancelled orders. |
| **Data Team (Historical Dashboard)** | **56.39%** | Technical legacy query counting observed completed orders ($843 / 1,495$). |

### Stakeholder Ownership
> **Which one should leadership publish, and who must own that decision?**

Leadership should publish the **Support Lead definition ($delay > 10$ minutes)** as the primary customer-facing Late Delivery Rate (**23.34%**). Publishing 56% inflates failure by counting trivial sub-minute variances.
The canonical metric owner must be the **Chief Operating Officer (COO)** in consensus with the **Head of Customer Experience**; technical teams must not decide policy thresholds.


In [ ]:
# Challenge 2: Calculate the late rate using different definitions

analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60.0

observed_delivered = analysis[analysis["actual_delivery_at"].notna()]
total_all = len(analysis)
late_gt_0 = (observed_delivered["delay_min"] > 0.0).sum()
late_gt_10 = (observed_delivered["delay_min"] > 10.0).sum()

comparison_df = pd.DataFrame([
    {
        "Stakeholder": "VP Operations (> 0m)",
        "Population": "Delivered with timestamp (n=1,495)",
        "Late Count": late_gt_0,
        "Total Orders": len(observed_delivered),
        "Late Rate": f"{(late_gt_0 / len(observed_delivered)):.2%}",
        "Business Context": "Strict contractual SLA; zero grace period"
    },
    {
        "Stakeholder": "Support Lead (> 10m)",
        "Population": "Delivered with timestamp (n=1,495)",
        "Late Count": late_gt_10,
        "Total Orders": len(observed_delivered),
        "Late Rate": f"{(late_gt_10 / len(observed_delivered)):.2%}",
        "Business Context": "Customer dissatisfaction threshold"
    },
    {
        "Stakeholder": "VP Operations (All)",
        "Population": "Total Order Intake (n=1,600)",
        "Late Count": late_gt_0,
        "Total Orders": total_all,
        "Late Rate": f"{(late_gt_0 / total_all):.2%}",
        "Business Context": "Includes cancelled orders in denominator"
    },
    {
        "Stakeholder": "Finance",
        "Population": "Delivered Orders Only (n=1,495)",
        "Late Count": late_gt_0,
        "Total Orders": len(observed_delivered),
        "Late Rate": f"{(late_gt_0 / len(observed_delivered)):.2%}",
        "Business Context": "Excludes cancelled and refunded orders"
    },
    {
        "Stakeholder": "Data Team (Dashboard)",
        "Population": "Historical Dashboard (n=1,495)",
        "Late Count": late_gt_0,
        "Total Orders": len(observed_delivered),
        "Late Rate": f"{(late_gt_0 / len(observed_delivered)):.2%}",
        "Business Context": "Source of the published 56% claim"
    }
])

display(comparison_df)


# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [ ]:
# Challenge 3: Validate categories without cleaning by instinct

print("=== 1. final_status ===")
print(orders["final_status"].value_counts(dropna=False))

print("\n=== 2. traffic_bucket ===")
print(orders["traffic_bucket"].value_counts(dropna=False))

print("\n=== 3. restaurant_status ===")
print(restaurant_status["status"].value_counts(dropna=False))

print("\n=== 4. support_tickets category ===")
print(tickets["category"].value_counts(dropna=False))

category_audit = pd.DataFrame([
    {"Field": "final_status", "Observed Anomaly": "'Delivered' (5), 'delivered' (1530)", "Normalization": "Safe (lowercase)", "Action": "Standardize string formatting"},
    {"Field": "traffic_bucket", "Observed Anomaly": "'HIGH' (1), 'high' (208)", "Normalization": "Safe (lowercase)", "Action": "Standardize string formatting"},
    {"Field": "restaurant_status", "Observed Anomaly": "'ready ' (2), 'READY' (1), 'handoff' (1), 'unknown' (1)", "Normalization": "Partial (strip & case safe)", "Action": "Confirm 'handoff' vs 'handed_off' with partner engineering"},
    {"Field": "support_tickets.category", "Observed Anomaly": "'Late Delivery' (1), 'late_delivery ' (1), 'ETA issue' (1)", "Normalization": "Partial (strip & case safe)", "Action": "Confirm 'ETA issue' classification with Support Lead"}
])
display(category_audit)


# Challenge 4 — Cross-source integrity

Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|
| `orders.restaurant_id` → `restaurants` | 100.0% | **PASS** | Zero referential orphan risk. |
| `orders.driver_id` → `drivers` | 100.0% | **PASS** | Zero referential orphan risk. |
| `tickets.order_id` → `orders` | 98.5% | **WARN** | 3 tickets (1.5%) have null `order_id`; untraceable to specific orders. |
| `restaurant_status.order_id` → `orders` | 100.0% (31.25% pop.) | **WARN** | Existing records link, but 68.75% of orders have no restaurant telemetry. |

### Is 1% Unmapped Acceptable?
> **If 1% is unmapped, is that acceptable?**

It depends on the business decision:
- For **weekly executive aggregate reporting**, 1% unmapped support tickets is acceptable.
- For **individual customer compensation or merchant penalty enforcement**, 1% unmapped is unacceptable because automated chargebacks cannot be executed without a verified `order_id`.


In [ ]:
# Challenge 4: Calculate mapping coverage for all relationships

cov_restaurant = orders_dedup["restaurant_id"].dropna().isin(restaurants["restaurant_id"]).mean()
cov_driver = orders_dedup["driver_id"].dropna().isin(drivers["driver_id"]).mean()
cov_tickets = tickets["order_id"].dropna().isin(orders_dedup["order_id"]).mean()
cov_status = restaurant_status["order_id"].dropna().isin(orders_dedup["order_id"]).mean()
status_population_share = len(restaurant_status["order_id"].dropna().unique()) / len(orders_dedup)

integrity_results = pd.DataFrame([
    {"Relationship": "orders.restaurant_id -> restaurants", "Coverage %": f"{cov_restaurant:.1%}", "Status": "PASS", "Risk": "Zero referential orphan risk"},
    {"Relationship": "orders.driver_id -> drivers", "Coverage %": f"{cov_driver:.1%}", "Status": "PASS", "Risk": "Zero referential orphan risk"},
    {"Relationship": "tickets.order_id -> orders (non-null)", "Coverage %": f"{cov_tickets:.1%}", "Status": "WARN", "Risk": "3 tickets have null order_id (unattributed)"},
    {"Relationship": "restaurant_status.order_id -> orders", "Coverage %": f"{cov_status:.1%}", "Status": "WARN", "Risk": f"Only {status_population_share:.1%} of orders have status records"}
])

display(integrity_results)


# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [ ]:
# Challenge 5: Freshness as an SLA question

orders_lifecycle = orders.drop_duplicates("order_id", keep="first").copy()
for c in ["created_at", "pickup_at", "actual_delivery_at"]:
    orders_lifecycle[c + "_dt"] = pd.to_datetime(orders_lifecycle[c], format="mixed", errors="coerce")

status_lifecycle = restaurant_status.copy()
status_lifecycle["status_dt"] = pd.to_datetime(status_lifecycle["last_updated_at"], format="mixed", errors="coerce")

m_status = status_lifecycle.merge(orders_lifecycle, on="order_id", how="inner")
m_status["lag_pickup_min"] = (m_status["status_dt"] - m_status["pickup_at_dt"]).dt.total_seconds() / 60.0
m_status["lag_created_min"] = (m_status["status_dt"] - m_status["created_at_dt"]).dt.total_seconds() / 60.0

post_pickup_cnt = (m_status["status_dt"] > m_status["pickup_at_dt"]).sum()
negative_lag_cnt = (m_status["status_dt"] < m_status["created_at_dt"]).sum()

print(f"Total restaurant status records merged: {len(m_status)}")
print(f"Status records timestamped AFTER driver pickup: {post_pickup_cnt} ({post_pickup_cnt / len(m_status):.1%})")
print(f"Status records timestamped BEFORE order creation: {negative_lag_cnt}")

print("\n--- Fit-for-Purpose SLA Evaluation ---")
sla_evaluation = pd.DataFrame([
    {"Use Case": "Weekly Analytics", "Fit Status": "WARN", "SLA Assessment": "Lags distort stage durations, but macro restaurant bottlenecks remain observable."},
    {"Use Case": "Live Customer ETA", "Fit Status": "FAIL", "SLA Assessment": "Unsafe; status updates arriving post-pickup corrupt real-time dynamic ETA engines."},
    {"Use Case": "Restaurant Accountability", "Fit Status": "FAIL", "SLA Assessment": "Unsafe; negative timestamps and post-pickup entries cannot defend contractual penalties."}
])
display(sla_evaluation)


# Challenge 6 — Build the validation gate

Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain | **WARN** | 3 duplicate order IDs (1,603 rows vs 1,600 unique) | Enforce primary key deduplication at ingestion. |
| Timestamp chronology | **FAIL** | 4 negative ETA durations; 5 negative transit durations | Reject or quarantine invalid timestamp rows in ETL. |
| KPI definition | **FAIL** | 4 conflicting stakeholder definitions (56% vs 23%) | Executive alignment to adopt >10 min threshold. |
| Category semantics | **WARN** | Casing noise ('Delivered') and unconfirmed statuses ('handoff') | Normalize representation; obtain partner engineering sign-off. |
| Cross-source mapping | **WARN** | 3 unlinked tickets; 68.75% missing restaurant status | Mandate order ID in support forms; expand kitchen tablet rollout. |
| Freshness | **FAIL** | 166 status updates recorded after driver pickup | Restrict status writes to live transactions; reject retroactive edits. |

### Final Publication Decision
> **Should leadership publish “Late Delivery Rate = 56%” today?**

**NO. Leadership must NOT publish “Late Delivery Rate = 56%” today.**

### Mandatory Remediation Gates Before Publication:
1. **Adopt Canonical Definition:** Formally ratify the **>10 minute delay threshold** (representing the true **23.34%** operational late rate).
2. **Eliminate Chronology Defects:** Fix clock synchronization and quarantine negative duration timestamps.
3. **Standardize Treatment of Incomplete Orders:** Establish formal operational accounting for the 37 unrecorded deliveries and 68 cancelled orders.


In [ ]:
# Challenge 6: Final validation report gate

validation_report = {
    "business_grain": "WARN",
    "timestamp_chronology": "FAIL",
    "kpi_definition": "FAIL",
    "category_semantics": "WARN",
    "cross_source_mapping": "WARN",
    "freshness": "FAIL",
    "publish_56_percent": "FAIL"
}

print("=" * 45)
print("FLASH EATS FDE DATA VALIDATION GATE REPORT")
print("=" * 45)
for check_item, gate_status in validation_report.items():
    print(f"{check_item.upper():<25}: {gate_status}")
print("=" * 45)


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**